# Installing FastF1 Library

In [1]:
pip install fastf1

Note: you may need to restart the kernel to use updated packages.


In [2]:
import fastf1
import pandas as pd
import os

os.makedirs('cache', exist_ok=True)  # Creates the folder if it doesn't exist
fastf1.Cache.enable_cache('cache')  # Enable cache to avoid redownloading data
fastf1.Cache.enable_cache('cache')  # Creates a "cache" folder locally

In [3]:
import warnings
warnings.filterwarnings("ignore")

In [5]:
YEAR = 2025
ROUNDS = list(range(1, 15))  # rounds 1–14

# --- helper to normalize session results ---
def session_results_df(session, round_no):
    res = session.results.copy()
    if res is None or len(res) == 0:
        return pd.DataFrame(columns=['round_no','raceId','driverId','driverName','teamName','position','points'])
    res = res.reset_index(drop=True)
    return pd.DataFrame({
        'round_no': round_no,
        'raceId': round_no,
        'driverId': res.get('DriverId', pd.Series([pd.NA]*len(res))),
        'driverName': res.get('Driver', pd.Series([pd.NA]*len(res))),
        'teamName': res.get('TeamName', pd.Series([pd.NA]*len(res))),
        'position': pd.to_numeric(res.get('Position', pd.Series([pd.NA]*len(res))), errors='coerce'),
        'points': pd.to_numeric(res.get('Points', pd.Series([0.0]*len(res))).fillna(0.0), errors='coerce')
    })

def race_session(year, rnd):
    sess = fastf1.get_session(year, rnd, 'Race')
    sess.load()
    return sess

def sprint_session(year, rnd):
    try:
        sess = fastf1.get_session(year, rnd, 'Sprint')
        sess.load()
        if sess.results is None or sess.results.empty:
            return None
        return sess
    except Exception:
        return None

# --- collect race + sprint results ---
race_results_all, sprint_results_all = [], []

for rnd in ROUNDS:
    rr = session_results_df(race_session(YEAR, rnd), rnd)
    race_results_all.append(rr)

    spr = sprint_session(YEAR, rnd)
    if spr is not None:
        sr = session_results_df(spr, rnd)
        sprint_results_all.append(sr)

race_results = pd.concat(race_results_all, ignore_index=True)
sprint_results = pd.concat(sprint_results_all, ignore_index=True) if sprint_results_all else pd.DataFrame(columns=race_results.columns)

# --- aggregate team totals ---
team_points_race = race_results.groupby('teamName', as_index=False)['points'].sum().rename(columns={'points':'racePoints'})
team_points_sprint = sprint_results.groupby('teamName', as_index=False)['points'].sum().rename(columns={'points':'sprintPoints'})
team_points = team_points_race.merge(team_points_sprint, on='teamName', how='outer').fillna(0)
team_points['totalPoints'] = team_points['racePoints'] + team_points['sprintPoints']
team_points['seasonPos'] = team_points['totalPoints'].rank(method='dense', ascending=False).astype(int)

# --- driver list per team ---
drivers_meta = pd.concat([
    race_results[['teamName','driverId','driverName']],
    sprint_results[['teamName','driverId','driverName']]
], ignore_index=True).dropna(subset=['teamName','driverName']).drop_duplicates()

team_drivers = (drivers_meta.groupby('teamName')
                .agg(driverCount=('driverId','nunique'),
                     driverNames=('driverName', lambda s: ', '.join(sorted(set(s)))))
                .reset_index())

# --- final team_2025.csv ---
team_2025 = (team_drivers.merge(team_points, on='teamName', how='outer')
             .fillna({'driverCount':0,'driverNames':''}))
team_2025['prevSeasonPos'] = pd.NA  # placeholder for 2024 comparison

team_2025 = team_2025[['teamName','driverCount','driverNames','racePoints','sprintPoints','totalPoints','seasonPos','prevSeasonPos']]
team_2025.sort_values(['seasonPos','teamName'], inplace=True)

print(team_2025.head(20))

core           INFO 	Loading data for Australian Grand Prix - Race [v3.6.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Fixed incorrect tyre stint information for driver '87'
core        WARNING 	Fixed incorrect tyre stint information for driver '30'
core        WARNING 	Fixed incorrect tyre stint information for driver '5'
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core    

          teamName  driverCount driverNames  racePoints  sprintPoints  \
5          McLaren          0.0                   523.0          36.0   
2          Ferrari          0.0                   237.0          23.0   
6         Mercedes          0.0                   222.0          14.0   
8  Red Bull Racing          0.0                   177.0          17.0   
9         Williams          0.0                    67.0           3.0   
1     Aston Martin          0.0                    48.0           4.0   
4      Kick Sauber          0.0                    51.0           0.0   
7     Racing Bulls          0.0                    41.0           4.0   
3     Haas F1 Team          0.0                    29.0           6.0   
0           Alpine          0.0                    19.0           1.0   

   totalPoints  seasonPos prevSeasonPos  
5        559.0          1          <NA>  
2        260.0          2          <NA>  
6        236.0          3          <NA>  
8        194.0          4   

In [6]:
team_2025.to_csv(r"D:\Masters Study Abroad\Self Study\Personal Projects\F1\Descriptive Analytics\team_2025.csv", index=False)

# To Procure Team Pitstop Table

In [11]:
YEAR = 2025
ROUNDS = list(range(1, 15))  # So far 1..14

def get_race_session(year, rnd):
    sess = fastf1.get_session(year, rnd, "Race")
    sess.load(laps=True, telemetry=False)
    return sess

def stops_from_laps(sess, rnd):
    laps = sess.laps
    if laps is None or laps.empty:
        return pd.DataFrame(columns=['round_no','raceId','driverName','teamName','stopSeconds'])
    # ensure columns exist
    for c in ['Driver','Team','PitInTime','PitOutTime','LapNumber']:
        if c not in laps.columns:
            laps[c] = pd.NA

    rows = []
    for drv, df in laps.groupby('Driver'):
        df = df.sort_values('LapNumber')
        in_times  = df['PitInTime'].dropna().tolist()
        out_times = df['PitOutTime'].dropna().tolist()
        team = df['Team'].dropna().iloc[0] if df['Team'].notna().any() else pd.NA
        n = min(len(in_times), len(out_times))
        for i in range(n):
            dur = (out_times[i] - in_times[i]).total_seconds()
            rows.append({
                'round_no': rnd,
                'raceId': rnd,
                'driverName': drv,
                'teamName': team,
                'stopSeconds': dur
            })
    return pd.DataFrame(rows)

# collect all rounds
all_stops = []
for rnd in ROUNDS:
    try:
        sess = get_race_session(YEAR, rnd)
        df = stops_from_laps(sess, rnd)
        all_stops.append(df)
    except Exception as e:
        print(f"[warn] round {rnd}: {e}")

stops = pd.concat(all_stops, ignore_index=True) if all_stops else pd.DataFrame(
    columns=['round_no','raceId','driverName','teamName','stopSeconds']
)

# --- FILTER bad stops at the stop-level ---
# keep: 0 < stopSeconds <= 1000
stops = stops[pd.to_numeric(stops['stopSeconds'], errors='coerce').notna()]
stops = stops[(stops['stopSeconds'] > 0) & (stops['stopSeconds'] <= 1000)]

# --- SEASON-TO-DATE per team ---
season_team = (stops
    .dropna(subset=['teamName'])
    .groupby('teamName', as_index=False)
    .agg(
        pitStops=('stopSeconds', 'count'),
        totalDuration=('stopSeconds', 'sum')
    )
)

season_team['avgDuration'] = season_team['totalDuration'] / season_team['pitStops']
season_team = season_team[['teamName','pitStops','totalDuration','avgDuration']].sort_values('teamName')
print(season_team.head(20))

core           INFO 	Loading data for Australian Grand Prix - Race [v3.6.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Fixed incorrect tyre stint information for driver '87'
core        WARNING 	Fixed incorrect tyre stint information for driver '30'
core        WARNING 	Fixed incorrect tyre stint information for driver '5'
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core        WARNING 	Driver 4 completed the race distance 00:00.022000 before the recorded end of the session.
core  

          teamName  pitStops  totalDuration  avgDuration
0           Alpine        41        956.397    23.326756
1     Aston Martin        51       1177.328    23.084863
2          Ferrari        56       1232.813    22.014518
3     Haas F1 Team        44       1065.183    24.208705
4      Kick Sauber        55       1243.466    22.608473
5          McLaren        54       1225.301    22.690759
6         Mercedes        54       1192.636    22.085852
7     Racing Bulls        42        958.981    22.832881
8  Red Bull Racing        43       1005.602    23.386093
9         Williams        45       1091.794    24.262089


In [13]:
season_team.to_csv(r"D:\Masters Study Abroad\Self Study\Personal Projects\F1\Descriptive Analytics\Team_Pitstop_2025.csv", index=False)

# To Procure Team Standings after Each Race

In [17]:
import pandas as pd
# load driver standings table
driver_standings = pd.read_csv(r"D:\Masters Study Abroad\Self Study\Personal Projects\F1\Descriptive Analytics\DriverStandings_2025.csv")

# team cumulative points after each round = sum of driver cumulativeTotalPoints
team_round = (driver_standings
              .groupby(['season','round','raceId','teamName'], as_index=False)
              .agg(cumPoints=('cumulativeTotalPoints','sum')))

# constructor position = rank based on cumPoints (descending)
team_round['champPos'] = (team_round
    .sort_values(['season','round','cumPoints'], ascending=[True, True, False])
    .groupby(['season','round'])['cumPoints']
    .rank(method='dense', ascending=False)
    .astype(int)
)

# also calculate teamPoints for that race (not cumulative) if needed
# by summing pointsAfterRace + sprintPoints
team_points = (driver_standings
               .groupby(['season','round','raceId','teamName'], as_index=False)
               .agg(teamPoints=('pointsAfterRace','sum'),
                    sprintPoints=('sprintPoints','sum')))
team_points['teamPoints'] = team_points['teamPoints'] + team_points['sprintPoints']

# merge race-level teamPoints back into cumulative table
team_season = team_round.merge(team_points[['season','round','raceId','teamName','teamPoints']],
                               on=['season','round','raceId','teamName'],
                               how='left')

# reorder + clean
team_season = team_season[['season','round','raceId','teamName','teamPoints','cumPoints','champPos']]
team_season = team_season.sort_values(['round','champPos','teamName'])

print(team_season.head(20))


    season  round    raceId         teamName  teamPoints  cumPoints  champPos
5     2025      1  2025_R01          McLaren          27         27         1
6     2025      1  2025_R01         Mercedes          27         27         1
8     2025      1  2025_R01  Red Bull Racing          18         18         2
9     2025      1  2025_R01         Williams          10         10         3
1     2025      1  2025_R01     Aston Martin           8          8         4
4     2025      1  2025_R01      Kick Sauber           6          6         5
2     2025      1  2025_R01          Ferrari           5          5         6
0     2025      1  2025_R01           Alpine           0          0         7
3     2025      1  2025_R01     Haas F1 Team           0          0         7
7     2025      1  2025_R01     Racing Bulls           0          0         7
15    2025      2  2025_R02          McLaren          78         78         1
16    2025      2  2025_R02         Mercedes          59        

In [18]:
team_season.to_csv(r"D:\Masters Study Abroad\Self Study\Personal Projects\F1\Descriptive Analytics\TeamStandings_2025.csv", index=False)

# To Procure Teams' Race Pace Table

In [19]:
YEAR = 2025
ROUNDS = list(range(1, 15))  # races completed so far

all_pace = []

for rnd in ROUNDS:
    try:
        sess = fastf1.get_session(YEAR, rnd, "Race")
        sess.load(laps=True, telemetry=False)
        laps = sess.laps
        
        if laps is None or laps.empty:
            continue

        # filter: quick laps & no pit in/out
        try:
            laps = laps.pick_quicklaps().pick_wo_box()
        except Exception:
            if "IsPitIn" in laps.columns: laps = laps[~laps["IsPitIn"].fillna(False)]
            if "IsPitOut" in laps.columns: laps = laps[~laps["IsPitOut"].fillna(False)]

        laps["lapTime_sec"] = laps["LapTime"].dt.total_seconds()

        if "Team" not in laps.columns:
            laps["Team"] = laps.get("TeamName", pd.NA)

        lap_clean = laps.dropna(subset=["Team","lapTime_sec"]).copy()

        team_pace = (lap_clean
            .groupby("Team", as_index=False)
            .agg(
                medianLap_sec=("lapTime_sec","median"),
                meanLap_sec=("lapTime_sec","mean"),
                bestLap_sec=("lapTime_sec","min"),
                lapsCount=("lapTime_sec","count")
            ))

        team_pace = team_pace.rename(columns={"Team":"teamName"})
        team_pace["round_no"] = rnd
        team_pace["raceId"] = rnd
        all_pace.append(team_pace)

        print(f"processed round {rnd}")

    except Exception as e:
        print(f"[warn] round {rnd} failed: {e}")

TeamPace_2025 = pd.concat(all_pace, ignore_index=True)

# order columns
TeamPace_2025 = TeamPace_2025[["round_no","raceId","teamName","medianLap_sec","meanLap_sec","bestLap_sec","lapsCount"]]

TeamPace_2025.to_csv("TeamPace_2025.csv", index=False)
print(TeamPace_2025.head(20))


core           INFO 	Loading data for Australian Grand Prix - Race [v3.6.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Fixed incorrect tyre stint information for driver '87'
core        WARNING 	Fixed incorrect tyre stint information for driver '30'
core        WARNING 	Fixed incorrect tyre stint information for driver '5'
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core        WARNING 	Driver 4 completed the race distance 00:00.022000 before the recorded end of the session.
core  

processed round 1


req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['81', '4', '63', '1', '31', '12', '23', '87', '18', '55', '6', '30', '7', '5', '27', '22', '14', '16', '44', '10']
core           INFO 	Loading data for Japanese Grand Prix - Race [v3.6.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...


processed round 2


req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '4', '81', '16', '63', '12', '44', '6', '23', '87', '14', '22', '10', '55', '7', '27', '30', '31', '5', '18']
core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.6.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...


...

processed round 12


core        WARNING 	Fixed incorrect tyre stint information for driver '16'
core        WARNING 	Fixed incorrect tyre stint information for driver '1'
core        WARNING 	Fixed incorrect tyre stint information for driver '63'
core        WARNING 	Fixed incorrect tyre stint information for driver '23'
core        WARNING 	Fixed incorrect tyre stint information for driver '44'
core        WARNING 	Fixed incorrect tyre stint information for driver '30'
core        WARNING 	Fixed incorrect tyre stint information for driver '5'
core        WARNING 	Fixed incorrect tyre stint information for driver '10'
core        WARNING 	Fixed incorrect tyre stint information for driver '87'
core        WARNING 	Fixed incorrect tyre stint information for driver '27'
core        WARNING 	Fixed incorrect tyre stint information for driver '22'
core        WARNING 	Fixed incorrect tyre stint information for driver '18'
core        WARNING 	Fixed incorrect tyre stint information for driver '31'
core        WA

processed round 13


req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['4', '81', '63', '16', '14', '5', '18', '30', '1', '12', '6', '44', '27', '55', '23', '31', '22', '43', '10', '87']


processed round 14
    round_no  raceId         teamName  medianLap_sec  meanLap_sec  \
0          1       1           Alpine        85.0200    85.020000   
1          1       1     Aston Martin        86.4195    86.419500   
2          1       1          Ferrari        86.0510    85.957000   
3          1       1     Haas F1 Team        87.1835    87.183500   
4          1       1      Kick Sauber        85.0065    85.419250   
5          1       1          McLaren        87.1260    85.757889   
6          1       1         Mercedes        85.6520    85.927500   
7          1       1     Racing Bulls        85.6700    85.670000   
8          1       1  Red Bull Racing        85.8245    85.304125   
9          1       1         Williams        85.8620    85.862000   
10         2       2           Alpine        98.1880    98.140264   
11         2       2     Aston Martin        98.5370    98.190945   
12         2       2          Ferrari        97.2300    97.179190   
13         2   

In [20]:
TeamPace_2025.to_csv(r"D:\Masters Study Abroad\Self Study\Personal Projects\F1\Descriptive Analytics\TeamPace_2025.csv", index=False)